# Cribl Search (%%cribl_search)

This notebook demonstrates the **%%cribl_search** cell magic. Put the magic on the **first line** of a code cell; everything below is either a **kql** (Cribl Search query language) expression, or even simply a language (**english**).

After the job finishes, rows are loaded into a **pandas** DataFrame in the kernel (default name `results_df`, or `var=`).


## Parameters (first line)

- **`var=name`** — Python variable for the DataFrame (default `results_df`).
- **`preview=true|false`** — Show the interactive result table in the cell output (default `true`).
- **`response=dataframe|json|raw`** — Result materialisation mode (`dataframe` is default).
- **`limit=N`** — Max rows to load into the DataFrame (Default is `0` = all rows returned).
- **`lang=kql|kusto|english`** — Query mode (`english` translates to KQL before search).
- **`dataset=name`** — Optional dataset hint for `lang=english` translations.
- **`earliest=` / `latest=`** — Time window passed to the Search API (defaults apply if omitted).


## Searching using KQL 
First, submit a **KQL** query, then visualise it in the following cell using the DataFrame from the original KQL query cell.


In [ ]:
%%cribl_search var=kql_df
dataset=cribl_search_sample | sort by _time desc | limit 1000

In [ ]:
# ### Prompt:
# Generate Python code that Using a kql_df dataframe, generate a diagram plotting top 10 srcaddr-dstaddr pairs by sum of bytes. Turn the bytes column to integer first.
import micropip
await micropip.install("seaborn")

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Ensure 'bytes' is integer
kql_df['bytes'] = kql_df['bytes'].astype(int)

# Create a pair column for clarity
kql_df['pair'] = kql_df['srcaddr'] + ' → ' + kql_df['dstaddr']

# Group by pair and sum bytes
top_pairs = (
    kql_df.groupby('pair')['bytes']
    .sum()
    .sort_values(ascending=False)
    .head(10)
    .reset_index()
)

# Plot
plt.figure(figsize=(12, 6))
sns.barplot(data=top_pairs, x='bytes', y='pair', palette='viridis')
plt.xlabel('Total Bytes')
plt.ylabel('SrcAddr → DstAddr Pair')
plt.title('Top 10 SrcAddr-DstAddr Pairs by Total Bytes')
plt.tight_layout()
plt.show()


## Searching using natural language

The two cells below show how a natural language query can be used in `%%cribl_search` cell. It will get translated into **KQL** which will be submitted as a query. The subsequent cell visualises the query results.


In [ ]:
%%cribl_search var=recent_df lang=english dataset=cribl_search_sample
Show the 2000 most recent entries


In [ ]:
import pandas as pd

recent_df.head(10)
recent_df.describe(include="all")


In [ ]:
import matplotlib.pyplot as plt

# Pick a numeric column if present
if len(recent_df) > 0:
    num = recent_df.select_dtypes(include=["number"]).columns
    if len(num) > 0:
        c = num[0]
        recent_df[c].head(200).plot(kind="line", title=f"Sample: {c}")
        plt.tight_layout()
        plt.show()
    else:
        print("No numeric columns — columns:", list(recent_df.columns))
else:
    print("Empty DataFrame")


## Retrieving external data from APIs using `externaldata`

This cell submits a multiline `externaldata` query exactly as written. Rows load into **`kql_df`** via `var=kql_df` on the magic line.


In [ ]:
%%cribl_search var=kql_df
externaldata
[
  "https://vpic.nhtsa.dot.gov/api/vehicles/getallmanufacturers?format=json"
]
with(
  dataField="Results"
)
